# Contrastive representation learning with NT-Xent

This lab explores the contrastive objective on synthetic embedding pairs. It demonstrates the loss mechanics, not a trained GeoAI model or benchmark.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'implementations').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'implementations/python/info-nce'))
from nt_xent import nt_xent_loss


## 1. Create paired views

Each row in view A is paired with the same row in view B. In a real study, the two views would come from a carefully chosen geospatial augmentation pipeline.

In [ ]:
rng=np.random.default_rng(7)
base=rng.normal(size=(8,16))
view_a=base+0.05*rng.normal(size=base.shape)
view_b=base+0.08*rng.normal(size=base.shape)
matched_loss=nt_xent_loss(view_a,view_b,temperature=0.15)
mismatched_loss=nt_xent_loss(view_a,np.roll(view_b,1,axis=0),temperature=0.15)
print('Matched positive pairs:',matched_loss)
print('Mismatched pairing:',mismatched_loss)


## 2. Inspect the similarity matrix

The positive partner for each anchor is the corresponding row in the other view. Every non-self embedding in the batch is a negative for that anchor.

In [ ]:
joined=np.concatenate([view_a,view_b],axis=0)
joined=joined/np.linalg.norm(joined,axis=1,keepdims=True)
similarity=joined@joined.T
plt.figure(figsize=(6,5)); plt.imshow(similarity,cmap='coolwarm',vmin=-1,vmax=1); plt.colorbar(label='Cosine similarity'); plt.title('Two-view embedding similarity'); plt.xlabel('Embedding index'); plt.ylabel('Embedding index'); plt.show()


## 3. Change the temperature

Temperature rescales similarities before softmax. Lower values sharpen the distribution and can make optimization more sensitive to hard negatives.

In [ ]:
for temperature in [0.05,0.1,0.2,0.5,1.0]:
    print(f'temperature={temperature:>4}: loss={nt_xent_loss(view_a,view_b,temperature=temperature):.4f}')


## Investigate

1. What defines a positive pair in this synthetic example?
2. Why do the other batch embeddings act as negatives?
3. How does temperature change the loss?
4. Propose geospatial augmentations for land-cover mapping and road-orientation tasks. Why should they differ?
5. Explain how overlapping tiles can leak across train and test partitions.
6. What additional components are required for a complete self-supervised training system?